# DRAI Radar Data Conversion and Inspection

This notebook converts raw mmWave radar `.npy` files into DRAI features for gesture recognition.


## Processing Pipeline

```

Raw .npy

  -> per-frame split
  -> Range FFT
  -> Doppler FFT
  -> RX/TX virtual antenna reshape
  -> remove DC/static Doppler bins
  -> select dynamic Doppler bins
  -> Angle FFT
  -> sum dynamic Doppler response
  -> DRAI tensor (frames, 32, 32)
  -> *_drai.npy

```

## Dynamic Range–Angle Images

A dynamic range–angle image (DRAI) is generated from raw mmWave radar signals through a sequence of Fourier-domain processing steps.

The range FFT extracts distance information, the Doppler FFT represents motion, and the angle FFT estimates the angle of arrival across virtual antennas. Near-zero Doppler bins are suppressed to reduce static clutter, and high-power dynamic bins are retained. Summing the selected responses along the Doppler dimension produces a two-dimensional DRAI frame.

A DRAI sequence therefore captures how motion-related range–angle responses evolve during a gesture. This compact representation reduces static environmental interference while preserving information that can be modelled by CNN, LSTM, and Transformer architectures.

In [46]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

## Data-Shape Convention

For each frame, the processing code expects data in this order:

`(range_samples, loops_per_frame, rx_channels, tx_chirps)`

For the sample file in this project, the full file shape is:

`(40, 128, 64, 4, 2)`

This corresponds to 40 frames, 128 ADC samples, 64 Doppler loops, 4 receive antennas, and 2 transmit chirps. The receive channels and transmit chirps are reshaped into 8 virtual-antenna channels before the angle FFT.

In [47]:
# Core DRAI generation functions.
# Optional diagnostic outputs expose intermediate values for verification.


def data_path_range(fftsize, input_data):
    """Apply Range FFT to one TX branch of one radar frame.

    Parameters
    ----------
    fftsize : int
        Number of FFT points used along the range/sample axis.
    input_data : np.ndarray
        Complex radar data with shape `(range_samples, loops_per_frame, rx_channels)`.
        In this notebook, a typical shape is `(128, 64, 4)`.

    Returns
    -------
    np.ndarray
        Range FFT output with shape `(fftsize, loops_per_frame, rx_channels)`.

    Notes
    -----
    For every RX antenna, the function:
    1. removes the slow-time mean at each loop position,
    2. applies a Hanning window along the range/sample axis,
    3. computes the FFT along the range/sample axis.
    """
    range_window = np.hanning(input_data.shape[0] + 2)[1:-1]
    num_lines = input_data.shape[1]
    num_antennas = input_data.shape[2]

    output = np.zeros((fftsize, num_lines, num_antennas), dtype=complex)

    for antenna_idx in range(num_antennas):
        # Select one receive antenna: shape `(range_samples, loops_per_frame)`.
        input_matrix = np.squeeze(input_data[:, :, antenna_idx])

        # Remove the mean along the range/sample dimension for each loop.
        # This helps suppress constant background components before FFT.
        input_matrix = input_matrix - np.mean(input_matrix, axis=0)

        # Windowing reduces FFT sidelobes.
        input_matrix = input_matrix * range_window[:, None]

        # Range FFT: axis 0 is the fast-time/range-sample axis.
        fft_output = np.fft.fft(input_matrix, fftsize, axis=0)
        output[:, :, antenna_idx] = fft_output

    return output


def data_path_doppler(fftsize, input_data):
    """Apply Doppler FFT to Range FFT output for one TX branch.

    Parameters
    ----------
    fftsize : int
        Number of FFT points used along the Doppler/loop axis.
    input_data : np.ndarray
        Range FFT data with shape `(range_bins, loops_per_frame, rx_channels)`.

    Returns
    -------
    np.ndarray
        Doppler FFT output with shape `(range_bins, fftsize, rx_channels)`.

    Notes
    -----
    The output is `fftshift`ed so that zero Doppler is centered. This makes it
    easier to zero out static or near-static bins later.
    """
    doppler_window = np.hanning(input_data.shape[1] + 2)[1:-1]
    num_range_bins = input_data.shape[0]
    num_antennas = input_data.shape[2]

    output = np.zeros((num_range_bins, fftsize, num_antennas), dtype=complex)

    for antenna_idx in range(num_antennas):
        # Select one receive antenna: shape `(range_bins, loops_per_frame)`.
        input_matrix = np.squeeze(input_data[:, :, antenna_idx])

        # Window along the Doppler/slow-time dimension.
        input_matrix = input_matrix * doppler_window[None, :]

        # Doppler FFT: axis 1 is the slow-time/loop axis.
        fft_output = np.fft.fft(input_matrix, fftsize, axis=1)
        fft_output = np.fft.fftshift(fft_output, axes=1)
        output[:, :, antenna_idx] = fft_output

    return output


def build_radar_config(frame_shape=None):
    """Create the radar-processing parameter dictionary used by this notebook.

    Parameters
    ----------
    frame_shape : tuple or None
        Optional per-frame shape `(range_samples, loops_per_frame, rx_channels, tx_chirps)`.
        When supplied, the config is aligned with the actual data shape.

    Returns
    -------
    dict
        A dictionary containing FFT sizes, radar dimensions, intermediate buffers,
        and DRAI selection parameters.
    """
    if frame_shape is None:
        range_samples = 128
        loops_per_frame = 64
        rx_channels = 4
        tx_chirps = 2
    else:
        if len(frame_shape) != 4:
            raise ValueError(
                "frame_shape must be `(range_samples, loops_per_frame, rx_channels, tx_chirps)`."
            )
        range_samples, loops_per_frame, rx_channels, tx_chirps = frame_shape

    radar_config = {
        # Data dimensions.
        "gAdcOneSampleSize": 4,
        "numAdcSamples": range_samples,
        "numRxChan": rx_channels,
        "numChirpsPerFrame": loops_per_frame * tx_chirps,
        "nLoopsIn1Frame": loops_per_frame,
        "nChirpsIn1Loop": tx_chirps,
        # FFT sizes used by the DRAI pipeline.
        "range_fftsize": 128,
        "doppler_fftsize": 64,
        "angle_fftsize": 32,
        # DRAI construction parameters.
        "num_range_bins_keep": 32,
        "dc_doppler_bin_start": 26,
        "dc_doppler_bin_stop": 36,
        "dynamic_threshold_ratio": 0.5,
    }

    radar_config["rangeFFTOut"] = np.zeros(
        (
            radar_config["range_fftsize"],
            radar_config["nLoopsIn1Frame"],
            radar_config["numRxChan"],
            radar_config["nChirpsIn1Loop"],
        ),
        dtype=complex,
    )
    radar_config["DopplerFFTOut"] = np.zeros(
        (
            radar_config["range_fftsize"],
            radar_config["doppler_fftsize"],
            radar_config["numRxChan"],
            radar_config["nChirpsIn1Loop"],
        ),
        dtype=complex,
    )

    return radar_config


def radar_process_frame(radar_config, frame_complex, return_debug=False):
    """Convert one raw radar frame into one DRAI image.

    Parameters
    ----------
    radar_config : dict
        Parameter dictionary returned by `build_radar_config`.
    frame_complex : np.ndarray
        One raw radar frame with shape
        `(range_samples, loops_per_frame, rx_channels, tx_chirps)`.
    return_debug : bool, default=False
        If True, also return intermediate metadata such as Doppler power,
        threshold, and selected Doppler bins.

    Returns
    -------
    np.ndarray
        DRAI image with shape `(num_range_bins_keep, angle_fftsize)`, usually `(32, 32)`.
    tuple[np.ndarray, dict]
        Returned only when `return_debug=True`.
    """
    range_fftsize = radar_config["range_fftsize"]
    doppler_fftsize = radar_config["doppler_fftsize"]
    angle_fftsize = radar_config["angle_fftsize"]
    num_rx_channels = radar_config["numRxChan"]
    tx_chirps = radar_config["nChirpsIn1Loop"]

    range_fft_out = radar_config["rangeFFTOut"]
    doppler_fft_out = radar_config["DopplerFFTOut"]

    # Process every TX chirp separately, then combine RX x TX as virtual antennas.
    for tx_idx in range(tx_chirps):
        tx_frame = frame_complex[:, :, :, tx_idx]
        range_fft_out[:, :, :, tx_idx] = data_path_range(range_fftsize, tx_frame)
        doppler_fft_out[:, :, :, tx_idx] = data_path_doppler(
            doppler_fftsize,
            range_fft_out[:, :, :, tx_idx],
        )

    # Reshape `(range, doppler, rx, tx)` into `(range, doppler, virtual_antennas)`.
    radar_data_pre_angle_fft = np.reshape(
        doppler_fft_out,
        (range_fftsize, doppler_fftsize, num_rx_channels * tx_chirps),
        order="F",
    )

    # Retain the configured near-range bins.
    keep_bins = radar_config["num_range_bins_keep"]
    radar_data_pre_angle_fft = radar_data_pre_angle_fft[:keep_bins, :, :].copy()

    # Zero near-DC Doppler bins to suppress static clutter and very slow background.
    dc_start = radar_config["dc_doppler_bin_start"]
    dc_stop = radar_config["dc_doppler_bin_stop"]
    radar_data_pre_angle_fft[:, dc_start:dc_stop, :] = 0

    # Estimate Doppler-bin power after DC suppression.
    # Shape after averaging/summing: `(doppler_fftsize,)`.
    doppler_power = np.sum(np.mean(np.abs(radar_data_pre_angle_fft), axis=2), axis=0)
    peak_value = np.max(doppler_power)
    threshold = peak_value * radar_config["dynamic_threshold_ratio"]

    # Select Doppler bins that likely contain dynamic gesture motion.
    selected_doppler_bins = np.where(doppler_power > threshold)[0]

    # Angle FFT across virtual antennas creates an angle dimension.
    radar_data_angle_range = np.fft.fft(radar_data_pre_angle_fft, angle_fftsize, axis=2)

    # Sum only selected dynamic Doppler bins to create the final DRAI image.
    drai_frame = np.squeeze(
        np.sum(np.abs(radar_data_angle_range[:, selected_doppler_bins, :]), axis=1)
    )

    # Shift angle FFT bins so the angle axis is centered for visualization/model input.
    drai_frame = np.fft.fftshift(drai_frame, axes=1)

    if return_debug:
        debug_info = {
            "range_fft_shape": range_fft_out.shape,
            "doppler_fft_shape": doppler_fft_out.shape,
            "pre_angle_fft_shape": radar_data_pre_angle_fft.shape,
            "doppler_power": doppler_power,
            "peak_value": float(peak_value),
            "threshold": float(threshold),
            "selected_doppler_bins": selected_doppler_bins,
            "num_selected_doppler_bins": int(selected_doppler_bins.size),
            "drai_shape": drai_frame.shape,
        }
        return drai_frame, debug_info

    return drai_frame


def generateDRAI(filename, return_debug=False):
    """Generate DRAI features for all frames in one `.npy` radar file.

    Parameters
    ----------
    filename : str or pathlib.Path
        Path to the raw radar `.npy` file. Expected shape is
        `(frames, range_samples, loops_per_frame, rx_channels, tx_chirps)`.
    return_debug : bool, default=False
        If True, return one debug dictionary per frame in addition to the DRAI array.

    Returns
    -------
    np.ndarray
        DRAI tensor with shape `(frames, 32, 32)` for the current configuration.
    tuple[np.ndarray, list[dict]]
        Returned only when `return_debug=True`.
    """
    filename = Path(filename)
    raw_data = np.load(filename)

    if raw_data.ndim != 5:
        raise ValueError(
            f"Expected raw data with 5 dimensions `(frames, range, loops, rx, tx)`, got {raw_data.shape}."
        )

    radar_config = build_radar_config(frame_shape=raw_data.shape[1:])

    processed_frames = []
    debug_records = []

    for frame_idx in range(raw_data.shape[0]):
        if return_debug:
            drai_frame, debug_info = radar_process_frame(
                radar_config,
                raw_data[frame_idx, :, :, :, :],
                return_debug=True,
            )
            debug_info["frame_idx"] = frame_idx
            debug_records.append(debug_info)
        else:
            drai_frame = radar_process_frame(
                radar_config,
                raw_data[frame_idx, :, :, :, :],
            )

        processed_frames.append(drai_frame)

    drai_data = np.stack(processed_frames, axis=0)

    if return_debug:
        return drai_data, debug_records

    return drai_data

In [48]:
# Inspection and visualization helpers.
# These functions are optional: they do not change the saved DRAI data.


def array_size_mb(array):
    """Return the memory size of a NumPy array in megabytes."""
    return array.nbytes / (1024 ** 2)


def print_array_summary(name, array):
    """Print shape, dtype, memory size, and magnitude statistics for an array.

    Complex radar arrays are summarized using magnitude values so that the output
    is easy to compare with real-valued DRAI arrays.
    """
    values = np.abs(array) if np.iscomplexobj(array) else array
    print(f"{name} shape: {array.shape}")
    print(f"{name} dtype: {array.dtype}")
    print(f"{name} memory: {array_size_mb(array):.2f} MB")
    print(
        f"{name} value range: min={values.min():.6g}, "
        f"mean={values.mean():.6g}, std={values.std():.6g}, max={values.max():.6g}"
    )
    print(f"{name} NaN count: {np.isnan(values).sum()}")


def inspect_raw_file(filename):
    """Load a raw `.npy` radar file and print useful metadata.

    Parameters
    ----------
    filename : str or pathlib.Path
        Raw radar file to inspect.

    Returns
    -------
    np.ndarray
        Loaded raw radar data.
    """
    filename = Path(filename)
    raw_data = np.load(filename)

    print(f"File: {filename}")
    print(f"File size: {filename.stat().st_size / (1024 ** 2):.2f} MB")
    print_array_summary("Raw radar data", raw_data)

    if raw_data.ndim == 5:
        frames, range_samples, loops, rx_channels, tx_chirps = raw_data.shape
        print(
            "Interpreted dimensions: "
            f"frames={frames}, range_samples={range_samples}, loops={loops}, "
            f"rx_channels={rx_channels}, tx_chirps={tx_chirps}"
        )
    else:
        print("Warning: expected a 5-D raw radar tensor.")

    return raw_data


def plot_raw_trace(raw_data, frame_idx=0, loop_idx=0, rx_idx=0, tx_idx=0):
    """Plot one complex raw radar trace along the range/sample axis.

    This helps check whether the input file contains non-empty complex data before
    running the full DRAI conversion.
    """
    trace = raw_data[frame_idx, :, loop_idx, rx_idx, tx_idx]
    x_axis = np.arange(trace.shape[0])

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(x_axis, trace.real, label="real")
    ax.plot(x_axis, trace.imag, label="imag", alpha=0.8)
    ax.plot(x_axis, np.abs(trace), label="magnitude", linewidth=2)
    ax.set_title(
        f"Raw trace | frame={frame_idx}, loop={loop_idx}, rx={rx_idx}, tx={tx_idx}"
    )
    ax.set_xlabel("Range/sample index")
    ax.set_ylabel("Value")
    ax.legend()
    plt.show()


def plot_doppler_selection(debug_info):
    """Visualize Doppler power and the selected dynamic Doppler bins for one frame."""
    doppler_power = debug_info["doppler_power"]
    threshold = debug_info["threshold"]
    selected_bins = debug_info["selected_doppler_bins"]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(doppler_power, marker="o", markersize=3, label="Doppler power")
    ax.axhline(threshold, color="red", linestyle="--", label="0.5 x peak threshold")

    if selected_bins.size > 0:
        ax.scatter(
            selected_bins,
            doppler_power[selected_bins],
            color="orange",
            zorder=3,
            label="selected bins",
        )

    ax.set_title(
        f"Doppler selection | frame={debug_info.get('frame_idx', '?')} | "
        f"selected={debug_info['num_selected_doppler_bins']} bins"
    )
    ax.set_xlabel("Doppler bin")
    ax.set_ylabel("Power")
    ax.legend()
    plt.show()


def plot_drai_frames(drai_data, frame_indices=None, max_frames=6, cmap="magma"):
    """Display selected DRAI frames as range-angle heatmaps.

    Parameters
    ----------
    drai_data : np.ndarray
        DRAI tensor with shape `(frames, range_bins, angle_bins)`.
    frame_indices : list[int] or None
        Specific frame indices to show. If None, frames are sampled evenly.
    max_frames : int, default=6
        Maximum number of frames to display when `frame_indices` is None.
    cmap : str, default="magma"
        Matplotlib colormap used for heatmaps.
    """
    if drai_data.ndim != 3:
        raise ValueError(f"Expected DRAI data with shape `(frames, range, angle)`, got {drai_data.shape}.")

    num_frames = drai_data.shape[0]
    if frame_indices is None:
        shown = min(max_frames, num_frames)
        frame_indices = np.linspace(0, num_frames - 1, shown, dtype=int).tolist()

    fig, axes = plt.subplots(1, len(frame_indices), figsize=(4 * len(frame_indices), 4), squeeze=False)

    for ax, frame_idx in zip(axes[0], frame_indices):
        image = ax.imshow(drai_data[frame_idx], aspect="auto", origin="lower", cmap=cmap)
        ax.set_title(f"Frame {frame_idx}")
        ax.set_xlabel("Angle bin")
        ax.set_ylabel("Range bin")
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    fig.tight_layout()
    plt.show()


def plot_drai_summary(drai_data):
    if drai_data.ndim != 3:
        raise ValueError(f"Expected DRAI data with shape `(frames, range, angle)`, got {drai_data.shape}.")

    frame_energy = np.sum(drai_data, axis=(1, 2))
    average_drai = np.mean(drai_data, axis=0)


In [49]:
def process_npy_files(input_dir, output_dir, visualize=False, max_preview_files=1):
    """Batch-convert raw radar `.npy` files into DRAI `.npy` files.

    Parameters
    ----------
    input_dir : str or pathlib.Path
        Folder containing raw `.npy` radar files.
    output_dir : str or pathlib.Path
        Folder where converted DRAI files will be saved.
    visualize : bool, default=True
        If True, show Doppler-selection and DRAI preview plots for the first few files.
    max_preview_files : int, default=1
        Maximum number of files to visualize during batch processing.

    Returns
    -------
    list[dict]
        One summary dictionary per successfully processed file.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")

    # Create the output directory when it does not already exist.
    output_dir.mkdir(parents=True, exist_ok=True)

    npy_files = sorted(input_dir.glob("*.npy"))
    if not npy_files:
        print(f"No .npy files found in: {input_dir}")
        return []

    print(f"Input directory: {input_dir}")
    print(f"Output directory: {output_dir}")
    # print(f"Found {len(npy_files)} .npy file(s).")

    summaries = []

    if visualize and file_index < max_preview_files:
        for file_index, input_file_path in enumerate(npy_files):
            print("\n" + "=" * 80)
            print(f"Processing file {file_index + 1}/{len(npy_files)}: {input_file_path.name}")

            # Inspect metadata through memory mapping before full processing.
            raw_preview = np.load(input_file_path, mmap_mode="r")
            print_array_summary("Raw preview", raw_preview)

            # Generate DRAI and retain per-frame diagnostics for verification plots.
            drai_data, debug_records = generateDRAI(input_file_path, return_debug=True)
            print_array_summary("DRAI output", drai_data)

            output_file_path = output_dir / f"{input_file_path.stem}_drai.npy"
            np.save(output_file_path, drai_data)

            summary = {
                "input_file": str(input_file_path),
                "output_file": str(output_file_path),
                "raw_shape": tuple(raw_preview.shape),
                "drai_shape": tuple(drai_data.shape),
                "drai_dtype": str(drai_data.dtype),
                "selected_bins_first_frame": int(debug_records[0]["num_selected_doppler_bins"]),
                "output_size_mb": output_file_path.stat().st_size / (1024 ** 2),
            }
            summaries.append(summary)

        print(f"Saved: {output_file_path}")
        print(f"Saved file size: {summary['output_size_mb']:.2f} MB")
        print(f"First-frame selected Doppler bins: {summary['selected_bins_first_frame']}")

        if visualize and file_index < max_preview_files:
            plot_doppler_selection(debug_records[0])
            plot_drai_summary(drai_data)
            plot_drai_frames(drai_data)

    print(f"Processing complete.")
    return summaries


## Run Conversion

The final cell converts every `.npy` file in `INPUT_DIRECTORY`, saves the DRAI files to `OUTPUT_DIRECTORY`, and displays verification plots for the first processed file.

In [50]:
INPUT_DIRECTORY = Path("/Users/blue/Desktop/Practicum/code/processing/raw")
OUTPUT_DIRECTORY = Path("/Users/blue/Desktop/Practicum/code/processing/drai")

processing_summaries = process_npy_files(
    INPUT_DIRECTORY,
    OUTPUT_DIRECTORY,
    visualize=False,
    max_preview_files=1,
)

processing_summaries

Input directory: /Users/blue/Desktop/Practicum/code/processing/raw
Output directory: /Users/blue/Desktop/Practicum/code/processing/drai
Processing complete.


[]